<a href="https://colab.research.google.com/github/hasmalee/aeropinn/blob/main/AeroPINNN_X_80steps_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# AeroPINN_X_v3_Colab.py
# Final stage-runner notebook script
# ==================================

from __future__ import annotations

import os
import gc
import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from airfoil import AirfoilGeometry, AirfoilFormatError
from parsec_fit import (
    fit_parsec_to_dat,
    fit_quality_report,
    preprocess_airfoil_for_parsec,
)
from network import MLP, load_checkpoint, save_checkpoint
from train_loop import (
    train_multi_airfoil,
    optimize_shape,
)
from postprocess import (
    validate_model_on_library,
    export_optimized_airfoil,
    export_run_report,
)
from xai import run_xai_suite

In [16]:
# ============================================================================
# CELL 1 ─ Environment
# ============================================================================
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [17]:
from google.colab import drive
drive.mount('/content/drive/AeroPINN')

Mounted at /content/drive/AeroPINN


In [18]:
# ============================================================================
# CELL 2 ─ User stage selection
# ============================================================================
# STAGE = 1 -> pilot
# STAGE = 2 -> research training
# STAGE = 3 -> optimize one selected airfoil using trained model
STAGE = 2

# folders
DATA_ROOT = Path("/content/airfoils")
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"

OUT_DIR = Path("/content/drive/MyDrive/AeroPINN")
CKPT_DIR = OUT_DIR / "checkpoints"
VAL_OUT_DIR = OUT_DIR / "validation"
OPT_OUT_DIR = OUT_DIR / "optimized_results"
XAI_OUT_DIR = OUT_DIR / "xai_outputs"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
VAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
OPT_OUT_DIR.mkdir(parents=True, exist_ok=True)
XAI_OUT_DIR.mkdir(parents=True, exist_ok=True)

RESUME_CHECKPOINT = None   # e.g. CKPT_DIR / "multi_airfoil_last.pt"



In [47]:
# ============================================================================
# CELL 3 ─ Config
# ============================================================================
if STAGE == 1:
    AOA_LIST = [-2, 0, 2, 4]
    RE_LIST = [5e4, 1e5]

    N_INT = 2000
    N_NEAR = 400
    N_WAKE = 400
    N_AIRFOIL = 600
    N_INLET = 400
    N_OUTLET = 400
    N_TOP = 400
    N_BOT = 400

    ADAM_STEPS = 1500
    LBFGS_STEPS = 0
    ADAPTIVE_EVERY = 300
    ADAPTIVE_CAP = 1000
    CANDIDATE_POOL = 4000
    GAUSS_PER_CENTER = 4
    RESIDUAL_PROBE_STRIDE = 6

elif STAGE == 2:
    AOA_LIST = [0.0, 4.0]
    RE_LIST = [1e6, 2e6]

    N_INT = 3000
    N_NEAR = 600
    N_WAKE = 600
    N_AIRFOIL = 800
    N_INLET = 500
    N_OUTLET = 500
    N_TOP = 500
    N_BOT = 500

    ADAM_STEPS = 5000
    LBFGS_STEPS = 10
    ADAPTIVE_EVERY = 100
    ADAPTIVE_CAP = 2000
    CANDIDATE_POOL = 6000
    GAUSS_PER_CENTER = 6
    RESIDUAL_PROBE_STRIDE = 4

else:  # STAGE 3
    AOA_LIST = [5.0]
    RE_LIST = [1e5]

    # Stage 3 is optimization, not large retraining
    N_INT = 1500
    N_NEAR = 300
    N_WAKE = 300
    N_AIRFOIL = 400
    N_INLET = 250
    N_OUTLET = 250
    N_TOP = 250
    N_BOT = 250

    ADAM_STEPS = 0
    LBFGS_STEPS = 0
    ADAPTIVE_EVERY = 0
    ADAPTIVE_CAP = 0
    CANDIDATE_POOL = 0
    GAUSS_PER_CENTER = 0
    RESIDUAL_PROBE_STRIDE = 1

XLIM = (-1.0, 2.0)
YLIM = (-1.0, 1.0)
NEAR_BAND = 0.003

LR_ADAM = 3e-4
LR_MIN = 1e-5
LR_LAMBDA = 3e-4
WEIGHT_DECAY = 1e-5

PRINT_EVERY = 100
SAVE_EVERY = 500
VAL_EVERY = 2

PARSEC_OPT_THRESHOLD = 4.5

In [36]:
# ============================================================================
# CELL 4 ─ Build train/val/test airfoil library
# ============================================================================
def load_airfoil_library(folder: Path, n_boundary: int = 1200):
    lib = []
    skipped = []

    for f in sorted(folder.glob("*.dat")):
        try:
            geom = AirfoilGeometry.from_dat(f, n_boundary=n_boundary)

            parsec_params = None
            fit_report = None
            fit_mse = None
            use_for_optimization = False

            try:
                pre = preprocess_airfoil_for_parsec(geom.boundary, n_surface=300)
                params, fit_mse = fit_parsec_to_dat(str(f), verbose=False)
                fit_report = fit_quality_report(pre["upper"], pre["lower"], params)

                if fit_report["max_error_pct"] <= PARSEC_OPT_THRESHOLD:
                    parsec_params = params
                    use_for_optimization = True

            except Exception as e_fit:
                fit_report = {
                    "quality": "POOR",
                    "max_error_pct": 100.0,
                    "mean_error_pct": 100.0,
                    "error_message": str(e_fit),
                }

            lib.append(
                {
                    "name": f.stem,
                    "path": str(f),
                    "geometry": geom,
                    "parsec_params": parsec_params,
                    "fit_report": fit_report,
                    "fit_mse": fit_mse,
                    "use_for_optimization": use_for_optimization,
                }
            )

        except Exception as e:
            skipped.append((f.name, str(e)))

    return lib, skipped


TRAIN_AIRFOILS, TRAIN_SKIPPED = load_airfoil_library(TRAIN_DIR)
VAL_AIRFOILS, VAL_SKIPPED = load_airfoil_library(VAL_DIR)
TEST_AIRFOILS, TEST_SKIPPED = load_airfoil_library(TEST_DIR)

print(f"Loaded {len(TRAIN_AIRFOILS)} training airfoils")
print(f"Loaded {len(VAL_AIRFOILS)} validation airfoils")
print(f"Loaded {len(TEST_AIRFOILS)} test airfoils")

if TRAIN_SKIPPED or VAL_SKIPPED or TEST_SKIPPED:
    print("\nSkipped files:")
    for item in TRAIN_SKIPPED + VAL_SKIPPED + TEST_SKIPPED:
        print(" ", item)

Loaded 28 training airfoils
Loaded 4 validation airfoils
Loaded 4 test airfoils


In [37]:
# ============================================================================
# CELL 5 ─ Model create or resume
# ============================================================================
model = MLP(
    in_dim=5,
    out_dim=4,
    hidden_dim=64,
    num_hidden_layers=16,
    activation="silu",
).to(device)

# if RESUME_CHECKPOINT is not None and Path(RESUME_CHECKPOINT).exists():
#     model = load_checkpoint(str(RESUME_CHECKPOINT), device=device)
#     print("Resumed model from:", RESUME_CHECKPOINT)

print(model)
print("Total parameters:", sum(p.numel() for p in model.parameters()))



MLP(5→4, 64×16, act=silu, params=64,068)
Total parameters: 64068


In [ ]:
# ============================================================================
# CELL 6 ─ Train stages 1/2
# ============================================================================
history = []
adaptive_weights = None

if STAGE in (1, 2):
    print("=" * 62)
    print(" STAGE 1: Pilot" if STAGE == 1 else " STAGE 2: Research")
    print(f" AdamW steps = {ADAM_STEPS}")
    print(f" L-BFGS steps = {LBFGS_STEPS}")
    print(f" Adaptive every = {ADAPTIVE_EVERY}")
    print(f" Adaptive cap   = {ADAPTIVE_CAP}")
    print("=" * 62)

    model, adaptive_weights, history = train_multi_airfoil(
        model=model,
        train_airfoils=TRAIN_AIRFOILS,
        val_airfoils=VAL_AIRFOILS,
        aoa_list=AOA_LIST,
        re_list=RE_LIST,
        xlim=XLIM,
        ylim=YLIM,
        N_int=N_INT,
        N_near=N_NEAR,
        N_airfoil=N_AIRFOIL,
        N_inlet=N_INLET,
        N_outlet=N_OUTLET,
        N_top=N_TOP,
        N_bot=N_BOT,
        N_wake=N_WAKE,
        near_band=NEAR_BAND,
        adam_steps=ADAM_STEPS,
        lr_adam=LR_ADAM,
        lr_min=LR_MIN,
        lr_lambda=LR_LAMBDA,
        weight_decay=WEIGHT_DECAY,
        refresh_every=ADAPTIVE_EVERY,
        adaptive_every=ADAPTIVE_EVERY,
        adaptive_cap=ADAPTIVE_CAP,
        candidate_pool=CANDIDATE_POOL,
        gauss_per_center=GAUSS_PER_CENTER,
        residual_probe_stride=RESIDUAL_PROBE_STRIDE,
        lbfgs_steps=LBFGS_STEPS,
        print_every=PRINT_EVERY,
        save_every=SAVE_EVERY,
        val_every=VAL_EVERY,
        out_dir=str(CKPT_DIR),
        device=device,
        seed=42,
        resume_checkpoint=str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None,
    )

    print("Training finished.")
    if adaptive_weights is not None:
        print("Final adaptive weights:", adaptive_weights.all_weights())

    save_checkpoint(model, str(CKPT_DIR / "final_model.pt"))


 STAGE 2: Research
 AdamW steps = 5000
 L-BFGS steps = 10
 Adaptive every = 100
 Adaptive cap   = 2000
train_multi_airfoil: 28 airfoils, 24 conditions, 10 blocks

BLOCK 1/10
  Airfoil : naca0009
  AoA     : 12.0 deg
  Re      : 2.000e+05
  Steps   : 500
  step   100  loss=1.0870e+03  pde=1.5820e+03  wall=3.9669e-01  lr=1.52e-04  λwall=0.67  λpde=0.69
  step   200  loss=1.9679e+02  pde=2.8561e+02  wall=8.9853e-02  lr=7.70e-05  λwall=0.66  λpde=0.69
  step   300  loss=8.3725e+00  pde=1.1228e+01  wall=3.4428e-02  lr=3.90e-05  λwall=0.66  λpde=0.69
  step   400  loss=2.4671e+02  pde=3.5850e+02  wall=1.8736e-02  lr=1.97e-05  λwall=0.66  λpde=0.69
  step   500  loss=9.9925e+01  pde=1.4469e+02  wall=1.2968e-02  lr=1.00e-05  λwall=0.65  λpde=0.69

BLOCK 2/10
  Airfoil : eppler378
  AoA     : 8.0 deg
  Re      : 5.000e+04
  Steps   : 500
  step   100  loss=2.2382e+01  pde=3.1464e+01  wall=5.6023e-03  lr=1.52e-04  λwall=0.68  λpde=0.69
  step   200  loss=3.8264e+00  pde=4.6789e+00  wall=1.3481e-

In [ ]:
import importlib
import train_loop


importlib.reload(train_loop)

<module 'train_loop' from '/content/train_loop.py'>

In [49]:
# ============================================================================
# HIGH-Re 100-COMBINATION CONFIG
# ============================================================================

STAGE = 2

# 25 airfoils × (2 AoA × 2 Re) = 100 combinations
AOA_LIST = [0.0, 4.0]
RE_LIST  = [1.0e6, 2.0e6]

# research-lite but high-Re safe config
N_INT = 3000
N_NEAR = 600
N_WAKE = 600
N_AIRFOIL = 800
N_INLET = 500
N_OUTLET = 500
N_TOP = 500
N_BOT = 500

ADAM_STEPS = 5000
LBFGS_STEPS = 10

ADAPTIVE_EVERY = 100
ADAPTIVE_CAP = 2000
CANDIDATE_POOL = 6000
GAUSS_PER_CENTER = 6
RESIDUAL_PROBE_STRIDE = 4

XLIM = (-1.0, 2.0)
YLIM = (-1.0, 1.0)
NEAR_BAND = 0.003

LR_ADAM = 3e-4
LR_MIN = 1e-5
LR_LAMBDA = 3e-4
WEIGHT_DECAY = 1e-5

PRINT_EVERY = 100
SAVE_EVERY = 500
VAL_EVERY = 2

BLOCK_STEPS = 500
BLOCKS_PER_RUN = 10
MAX_TOTAL_COMBOS = 100

In [ ]:
import importlib
import train_loop
importlib.reload(train_loop)

from train_loop import train_multi_airfoil
print(train_multi_airfoil)

<function train_multi_airfoil at 0x7d8a59418b80>


In [ ]:
import importlib
import losses
import train_loop

importlib.reload(losses)
importlib.reload(train_loop)

<module 'train_loop' from '/content/train_loop.py'>

In [22]:
from pathlib import Path
import json

state_file = Path(OUT_DIR) / "combo_state.json"
state = json.loads(state_file.read_text())
print(state)

{'next_block_index': 40, 'total_selected_combos': 100, 'completed_blocks_this_run': 10, 'finished_all_selected_combos': False}


In [58]:
from pathlib import Path
import json

state_file = Path("/content/drive/MyDrive/AeroPINN/combo_state.json")
state = json.loads(state_file.read_text())

state["next_block_index"] = 70
state["finished_all_selected_combos"] = False

state_file.write_text(json.dumps(state, indent=2))
print("combo_state.json fixed for resume from human block 71")
print(state)

combo_state.json fixed for resume from human block 71
{'next_block_index': 70, 'total_selected_combos': 100, 'completed_blocks_this_run': 10, 'finished_all_selected_combos': False}


In [24]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [59]:
from pathlib import Path
import json
import torch

STATE_FILE = Path("/content/drive/MyDrive/AeroPINN/combo_state.json")
RESUME_CHECKPOINT = Path("/content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt")
OUT_DIR = Path("/content/drive/MyDrive/AeroPINN")

print("Checkpoint exists:", RESUME_CHECKPOINT.exists())
print("State exists:", STATE_FILE.exists())

state = json.loads(STATE_FILE.read_text())
state["next_block_index"] = 70
state["finished_all_selected_combos"] = False
STATE_FILE.write_text(json.dumps(state, indent=2))
print("Updated state:", state)

ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
print("Checkpoint keys:", ckpt.keys())

model.load_state_dict(ckpt["model_state"])
print("Loaded model_state from:", RESUME_CHECKPOINT)

history = ckpt.get("history", {})
print("Saved step:", ckpt.get("step"))
print("Saved block_index:", ckpt.get("block_index"))

Checkpoint exists: True
State exists: True
Updated state: {'next_block_index': 70, 'total_selected_combos': 100, 'completed_blocks_this_run': 10, 'finished_all_selected_combos': False}
Checkpoint keys: dict_keys(['step', 'block_index', 'model_state', 'optimizer_state', 'scheduler_state', 'lambda_optimizer_state', 'loss_weights_state', 'adaptive_sampler_state', 'history', 'extra'])
Loaded model_state from: /content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt
Saved step: 5000
Saved block_index: 70


In [39]:
# ============================================================================
# CELL ─ High-Re run settings
# ============================================================================

BLOCK_STEPS = 500
BLOCKS_PER_RUN = 10
MAX_TOTAL_COMBOS = 100

print("BLOCK_STEPS =", BLOCK_STEPS)
print("BLOCKS_PER_RUN =", BLOCKS_PER_RUN)
print("MAX_TOTAL_COMBOS =", MAX_TOTAL_COMBOS)

BLOCK_STEPS = 500
BLOCKS_PER_RUN = 10
MAX_TOTAL_COMBOS = 100


In [40]:
# Check which required variables are still missing
required_names = [
    "TRAIN_AIRFOILS", "VAL_AIRFOILS",
    "AOA_LIST", "RE_LIST",
    "XLIM", "YLIM",
    "N_INT", "N_NEAR", "N_AIRFOIL", "N_INLET", "N_OUTLET", "N_TOP", "N_BOT", "N_WAKE",
    "NEAR_BAND",
    "ADAM_STEPS", "LBFGS_STEPS",
    "LR_ADAM", "LR_MIN", "LR_LAMBDA", "WEIGHT_DECAY",
    "ADAPTIVE_EVERY", "ADAPTIVE_CAP",
    "CANDIDATE_POOL", "GAUSS_PER_CENTER", "RESIDUAL_PROBE_STRIDE",
    "PRINT_EVERY", "SAVE_EVERY", "VAL_EVERY",
    "train_multi_airfoil"
]

missing = [name for name in required_names if name not in globals()]
print("Missing:", missing)

Missing: []


In [60]:
# ============================================================================
# CELL ─ Run 10 new high-Re combinations
# ============================================================================

print("=" * 62)
print(" STAGE 2: High-Re 100-combination training")
print(f" AdamW steps = {ADAM_STEPS}")
print(f" L-BFGS steps = {LBFGS_STEPS}")
print(f" Adaptive every = {ADAPTIVE_EVERY}")
print(f" Adaptive cap   = {ADAPTIVE_CAP}")
print(f" Blocks/run     = {10}")
print(f" Max combos     = {100}")
print(f" AoA list       = {AOA_LIST}")
print(f" Re list        = {RE_LIST}")
print("=" * 62)

model, adaptive_weights, history = train_multi_airfoil(
    model=model,
    train_airfoils=TRAIN_AIRFOILS,
    val_airfoils=VAL_AIRFOILS,
    aoa_list=[0.0, 4.0],
    re_list=[1e6, 2e6],
    xlim=XLIM,
    ylim=YLIM,
    N_int=N_INT,
    N_near=N_NEAR,
    N_airfoil=N_AIRFOIL,
    N_inlet=N_INLET,
    N_outlet=N_OUTLET,
    N_top=N_TOP,
    N_bot=N_BOT,
    N_wake=N_WAKE,
    near_band=NEAR_BAND,
    adam_steps=ADAM_STEPS,
    lr_adam=LR_ADAM,
    lr_min=LR_MIN,
    lr_lambda=LR_LAMBDA,
    weight_decay=WEIGHT_DECAY,
    refresh_every=ADAPTIVE_EVERY,
    adaptive_every=ADAPTIVE_EVERY,
    adaptive_cap=ADAPTIVE_CAP,
    candidate_pool=CANDIDATE_POOL,
    gauss_per_center=GAUSS_PER_CENTER,
    residual_probe_stride=RESIDUAL_PROBE_STRIDE,
    lbfgs_steps=LBFGS_STEPS,
    print_every=PRINT_EVERY,
    save_every=SAVE_EVERY,
    val_every=VAL_EVERY,
    out_dir=Path("/content/drive/MyDrive/AeroPINN"),
    device=device,
    seed=42,
    block_steps=500,
    blocks_per_run=10,
    max_total_combos=100,
    resume_checkpoint=str(RESUME_CHECKPOINT),
)

print("\nBatch finished.")
if adaptive_weights is not None:
    print("Final adaptive weights:", adaptive_weights.all_weights())
print("History rows:", len(history))

 STAGE 2: High-Re 100-combination training
 AdamW steps = 5000
 L-BFGS steps = 10
 Adaptive every = 100
 Adaptive cap   = 2000
 Blocks/run     = 10
 Max combos     = 100
 AoA list       = [0.0, 4.0]
 Re list        = [1000000.0, 2000000.0]
train_multi_airfoil: 28 airfoils, 4 conditions, selected combos=100, running blocks 70..79

BLOCK 71/100
  Airfoil : s809
  AoA     : 4.0 deg
  Re      : 1.000e+06
  Steps   : 500
  step   100  loss=1.0253e-04  pde=5.7658e-05  wall=2.6729e-07  lr=1.52e-04  λwall=0.13  λpde=0.69
  step   200  loss=5.1957e-05  pde=9.9212e-06  wall=6.4038e-09  lr=7.70e-05  λwall=0.13  λpde=0.81
  step   300  loss=6.1704e-05  pde=7.9823e-06  wall=6.9076e-09  lr=3.90e-05  λwall=0.13  λpde=0.78
  step   400  loss=7.6089e-05  pde=9.7179e-06  wall=7.3451e-09  lr=1.97e-05  λwall=0.13  λpde=0.86
  step   500  loss=7.4141e-05  pde=8.2210e-06  wall=7.2830e-09  lr=1.00e-05  λwall=0.13  λpde=0.89

BLOCK 72/100
  Airfoil : naca64_012
  AoA     : 0.0 deg
  Re      : 1.000e+06
  Step

In [ ]:
import json
from pathlib import Path
import torch

def save_batch_checkpoint_permanent(
    model,
    adaptive_weights,
    history,
    out_dir,
    next_block_index,
    total_selected_combos,
    tag="batch_checkpoint",
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path = out_dir / f"{tag}.pt"
    state_path = out_dir / "combo_state.json"
    history_path = out_dir / "history.json"

    payload = {
        "model_state": model.state_dict(),
        "loss_weights_state": adaptive_weights.state_dict() if adaptive_weights is not None else None,
        "block_index": int(next_block_index),
        "history": history,
        "extra": {
            "next_block_index": int(next_block_index),
            "total_selected_combos": int(total_selected_combos),
        },
    }
    torch.save(payload, ckpt_path)

    state_path.write_text(
        json.dumps(
            {
                "next_block_index": int(next_block_index),
                "total_selected_combos": int(total_selected_combos),
                "finished_all_selected_combos": bool(next_block_index >= total_selected_combos),
            },
            indent=2,
        )
    )

    history_path.write_text(json.dumps(history, indent=2))

    print("Saved permanent checkpoint to:", ckpt_path)
    print("Saved combo state to        :", state_path)
    print("Saved history to            :", history_path)

In [ ]:
save_batch_checkpoint_permanent(
    model=model,
    adaptive_weights=adaptive_weights,
    history=history,
    out_dir=OUT_DIR,
    next_block_index=10,          # change this to the real next index
    total_selected_combos=100,
    tag="multi_airfoil_last",
)

Saved permanent checkpoint to: /content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt
Saved combo state to        : /content/drive/MyDrive/AeroPINN/combo_state.json
Saved history to            : /content/drive/MyDrive/AeroPINN/history.json


In [ ]:
import json
from pathlib import Path

state_file = Path(OUT_DIR) / "combo_state.json"
state = json.loads(state_file.read_text())

next_block_index = state["next_block_index"]
total_selected_combos = state["total_selected_combos"]

save_batch_checkpoint_permanent(
    model=model,
    adaptive_weights=adaptive_weights,
    history=history,
    out_dir=OUT_DIR,
    next_block_index=next_block_index,
    total_selected_combos=total_selected_combos,
    tag="multi_airfoil_last",
)

Saved permanent checkpoint to: /content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt
Saved combo state to        : /content/drive/MyDrive/AeroPINN/combo_state.json
Saved history to            : /content/drive/MyDrive/AeroPINN/history.json


In [ ]:
#load it later

# import torch
# from pathlib import Path

# ckpt_path = Path(OUT_DIR) / "multi_airfoil_last.pt"
# data = torch.load(ckpt_path, map_location=device, weights_only=False)

# model.load_state_dict(data["model_state"])

# if adaptive_weights is not None and data.get("loss_weights_state") is not None:
#     adaptive_weights.load_state_dict(data["loss_weights_state"])

# history = data.get("history", [])
# next_block_index = data.get("block_index", 0)

# print("Loaded checkpoint from:", ckpt_path)
# print("Next block index:", next_block_index)
# print("History rows:", len(history))